In [294]:
from fifa.matches import wc_matches, check_match_counts, latest_year
from fifa.ingest import load_rankings, load_rankings_json
import pandas as pd
from fifa import config

In [295]:

matches = wc_matches()
rankings= load_rankings()

In [296]:

rankings = pd.concat(
    [
        load_rankings(),                                  # CSV, through 2018
        load_rankings_json(config.FIFA_RANKING_JSON_2022, "2022-10-06"),      # 2022
        load_rankings_json(config.FIFA_RANKING_JSON_2026, "2026-06-05"),      # 2026
    ],
    ignore_index=True,
)

In [297]:
# Earliest match date per tournament
wc_start_date = matches.groupby("year")["date"].min()
wc_years = list(wc_start_date.index)
wc_start_date

year
1994   1994-06-17
1998   1998-06-10
2002   2002-05-31
2006   2006-06-09
2010   2010-06-11
2014   2014-06-12
2018   2018-06-14
2022   2022-11-20
2026   2026-06-11
Name: date, dtype: datetime64[us]

In [298]:
# Tag each ranking row with its year, then list the snapshot dates available per year.
rankings["year"] = rankings["rank_date"].dt.year
snapshot_dates = rankings.groupby("year")["rank_date"].unique()

# For each WC year, the latest snapshot on or before kickoff
closest_rank_date = {}
for year in wc_years:
    if year not in snapshot_dates:
        continue
    eligible = [d for d in snapshot_dates[year] if d <= wc_start_date[year]]
    if eligible:
        closest_rank_date[year] = max(eligible)
closest_rank_date

{1994: Timestamp('1994-06-14 00:00:00'),
 1998: Timestamp('1998-05-20 00:00:00'),
 2002: Timestamp('2002-05-15 00:00:00'),
 2006: Timestamp('2006-05-17 00:00:00'),
 2010: Timestamp('2010-05-26 00:00:00'),
 2014: Timestamp('2014-06-05 00:00:00'),
 2018: Timestamp('2018-06-07 00:00:00'),
 2022: Timestamp('2022-10-06 00:00:00'),
 2026: Timestamp('2026-06-05 00:00:00')}

In [299]:
# Map each year to its chosen snapshot date
rankings["snapshot_date"] = rankings["year"].map(closest_rank_date)

# Keep only the snapshot rows -> one rank per (year, country).
rank_lookup = rankings[rankings["rank_date"] == rankings["snapshot_date"]][
    ["year", "country_full", "rank"]
].copy()

In [300]:
rank_lookup[rank_lookup["country_full"].str.contains("Türkiye")]

,year,country_full,rank
57837,2022,Türkiye,45
58025,2026,Türkiye,22


In [301]:
# FIFA ranking name -> results-dataset name.
name_fixes_rankings = {
    "Korea Republic": "South Korea",
    "Korea DPR": "North Korea",
    "China PR": "China",
    "USA": "United States",
    "IR Iran": "Iran",
    "Czechia": "Czech Republic",
    "Congo DR": "DR Congo",
    "Türkiye": "Turkey",
    "Cabo Verde": "Cape Verde",
    "Cape Verde Islands": "Cape Verde",
    "Côte d'Ivoire": "Ivory Coast",
    "Serbia and Montenegro":"Serbia"

}
rank_lookup["country_full"] = rank_lookup["country_full"].replace(name_fixes_rankings)

In [302]:
# Left-merge the snapshot rank onto the home team, then the away team.
matches = matches.merge(
    rank_lookup.rename(columns={"country_full": "home_team", "rank": "home_team_rank"}),
    on=["year", "home_team"],
    how="left",
)
matches = matches.merge(
    rank_lookup.rename(columns={"country_full": "away_team", "rank": "away_team_rank"}),
    on=["year", "away_team"],
    how="left",
)

In [ ]:
# Drop all entries whose rank is not available - at this point that's only Iran(1998) & Serbia(1998)
matches = matches[~(matches["home_team_rank"].isna() | matches["away_team_rank"].isna())]

Index(['date', 'year', 'stage', 'home_team', 'away_team', 'home_score',
       'away_score', 'outcome', 'winner', 'decided_by_shootout',
       'is_host_match', 'neutral', 'city', 'country', 'home_team_rank',
       'away_team_rank'],
      dtype='str')